## 1. Importação de Bibliotecas

In [ ]:
import warnings
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

warnings.filterwarnings("ignore")

## 2. Configuração do Ambiente

In [ ]:
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

BASE_DIR = Path(".")
FIGURES_DIR = BASE_DIR / "figures"
TABLES_DIR = BASE_DIR / "tables"

FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.3

## 3. Funções Auxiliares

In [ ]:
def load_excel(filename: str) -> pd.DataFrame:
    path = BASE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    return pd.read_excel(path)


def save_figure(fig: plt.Figure, filename: str) -> None:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")

## 4. Carregamento dos Dados

In [ ]:
df_ativos = load_excel("Estudantes_ativos_EP_2025.xlsx")
df_inativos = load_excel("Estudantes_inativos_EP_2025.xlsx")
df_concat = load_excel("Estudantes_EP_2025_concat.xlsx")

df_inativos = df_inativos[df_inativos['Ano_Ingresso'] >= 2012]
df_concat = df_concat[df_concat['Ano_Ingresso'] >= 2012]

## 5. Preparação dos Dados

In [ ]:
features_logit = [
    "IRA",
    "Perfil",
    "Porcentagem_Concluido_SIGA",
    "Porcentagem_Inscrito",
    "Porcentagem_Aprovado",
    "Porcentagem_Reprovado",
    "Numero_Horas_Inscritas_Resultado",
    "Horas_Aprovadas",
]
features = features_logit

TARGET_COL = "Evadido_flag"

df = df_concat.copy()

num_cols = ["Ano_Ingresso", "Período_Ingresso","IRA","Porcentagem_Concluido_SIGA", "Porcentagem_Inscrito","Porcentagem_Aprovado", "Porcentagem_Reprovado","Numero_Horas_Exigidas_Curso","Numero_Horas_Inscritas_Resultado","Numero_Horas_Inscritas_Periodo_Atual","Horas_Aprovadas","ano_conclusao_ensino_medio","Ano_Egresso", "Periodo_Egresso", "Tempo_Evasao",]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.strip()
if "Evadido_flag" not in df.columns:
    def class_situacao_flag(row):
        status = str(row.get("Status", "")).strip()
        if status in ["Cursando", "Candidato à Formatura"]:
            return 0
        elif pd.isna(row.get("Ano_Egresso")):
            return 0
        else:
            return 1
    df["Evadido_flag"] = df.apply(class_situacao_flag, axis=1)

def class_situacao(row):
    status = str(row.get("Status", "")).strip()
    if status in ["Cursando", "Candidato à Formatura"]:
        return "Ativo"
    elif pd.isna(row.get("Ano_Egresso")):
        return "Ativo"
    else:
        return "Inativo"

df["Situacao"] = df.apply(class_situacao, axis=1)
df["Coorte"] = df["Ano_Ingresso"]

print("\nContagem por Status:")
print(df["Status"].value_counts())

print("\nContagem por Situacao:")
print(df["Situacao"].value_counts())

print("\nContagem por Evadido_flag (0 = não evadido, 1 = evadido):")
print(df["Evadido_flag"].value_counts(dropna=False))

features = [f for f in features if f in df.columns]

df_model = df.dropna(subset=features + ["Evadido_flag"]).copy()

## 6. Funções de Avaliação e Preparação de Features

In [ ]:
def avaliar_features_logit(
    df_model,
    features_logit,
    target_col,
    max_missing=0.4,
    corr_threshold=0.9,
    prefixo_arquivo="logit",
):
    presentes = [f for f in features_logit if f in df_model.columns]
    
    if not presentes:
        return [], features_logit, None, None
    
    sub = df_model[presentes].apply(pd.to_numeric, errors="coerce")
    
    missing_prop = sub.isnull().sum() / len(sub)
    validas = missing_prop[missing_prop <= max_missing].index.tolist()
    
    if len(validas) < 2:
        return validas, [f for f in presentes if f not in validas], missing_prop, None
    
    corr_mat = sub[validas].corr().abs()
    
    to_drop = set()
    for i in range(len(corr_mat.columns)):
        for j in range(i + 1, len(corr_mat.columns)):
            if corr_mat.iloc[i, j] > corr_threshold:
                to_drop.add(corr_mat.columns[j])
    
    selecionadas = [f for f in validas if f not in to_drop]
    descartadas = [f for f in presentes if f not in selecionadas]
    
    return selecionadas, descartadas, missing_prop, corr_mat


def preparar_dados_logit(
    df_model,
    features_usadas,
    target_col,
    test_size=0.3,
    random_state=42,
):

    X = df_model[features_usadas]
    y = df_model[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return {
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler,
    }

## 7. Função de Ajuste de Regressão Logística

In [ ]:
def ajustar_regressao_logistica(
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    random_state=42,
    prefixo_arquivo="logit",
):
    logit = LogisticRegression(random_state=random_state, max_iter=1000)
    logit.fit(X_train_scaled, y_train)
    
    y_pred = logit.predict(X_test_scaled)
    y_proba = logit.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"Acurácia: {acc:.3f}")
    print(f"AUC ROC: {auc:.3f}")
    print("\nMatriz de confusão:")
    print(cm)
    
    print("\nRelatório de classificação:")
    print(classification_report(y_test, y_pred, digits=3))
    
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    
    fig_roc, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--")
    ax.set_xlabel("Falso positivo")
    ax.set_ylabel("Verdadeiro positivo")
    ax.legend()
    plt.tight_layout()
    save_figure(fig_roc, f"{prefixo_arquivo}_curva_roc.png")
    plt.show()
    
    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Não evadido (0)", "Evadido (1)"],
        yticklabels=["Não evadido (0)", "Evadido (1)"],
        ax=ax_cm,
    )
    ax_cm.set_xlabel("Classe predita")
    ax_cm.set_ylabel("Classe verdadeira")
    plt.tight_layout()
    save_figure(fig_cm, f"{prefixo_arquivo}_matriz_confusao.png")
    plt.show()
    
    return {
        "model": logit,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "acc": acc,
        "auc": auc,
        "cm": cm,
    }

## 8. Função para Gerar Tabela e Figura de Coeficientes

In [ ]:
def gerar_tabela_e_fig_coeficientes(logit, features_usadas, prefixo_arquivo="logit"):
    coef_df = pd.DataFrame({
        "Feature": features_usadas,
        "Coeficiente": logit.coef_[0]
    })
    
    coef_df_sorted = coef_df.sort_values("Coeficiente", ascending=False)
    
    display(coef_df_sorted)
    
    coef_df_sorted.to_excel(TABLES_DIR / f"{prefixo_arquivo}_coeficientes.xlsx", index=False)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(data=coef_df_sorted, x="Coeficiente", y="Feature", ax=ax)
    ax.set_xlabel("Coeficiente (log-odds)")
    ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
    plt.tight_layout()
    save_figure(fig, f"{prefixo_arquivo}_coeficientes.png")
    plt.show()
    
    return coef_df, coef_df_sorted

## 9. Seleção de Features

In [ ]:
MAX_MISSING = 0.4
CORR_THRESHOLD = 0.9

features_usadas, features_descartadas, missing_prop, corr_mat = avaliar_features_logit(
    df_model=df_model,
    features_logit=features_logit,
    target_col=TARGET_COL,
    max_missing=MAX_MISSING,
    corr_threshold=CORR_THRESHOLD,
    prefixo_arquivo="logit",
)

print("Features selecionadas para o modelo:", features_usadas)
print("Features descartadas automaticamente:", features_descartadas)

## 10. Preparação dos Dados para Modelagem

In [ ]:
dados_logit = preparar_dados_logit(
    df_model=df_model,
    features_usadas=features_usadas,
    target_col=TARGET_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train_scaled = dados_logit["X_train_scaled"]
X_test_scaled = dados_logit["X_test_scaled"]
X_test = dados_logit["X_test"]
y_train = dados_logit["y_train"]
y_test = dados_logit["y_test"]

## 11. Treinamento e Avaliação - Regressão Logística

In [ ]:
resultado_logit = ajustar_regressao_logistica(
    X_train_scaled=X_train_scaled,
    X_test_scaled=X_test_scaled,
    y_train=y_train,
    y_test=y_test,
    random_state=RANDOM_STATE,
    prefixo_arquivo="logit",
)

logit = resultado_logit["model"]
y_pred = resultado_logit["y_pred"]
y_proba = resultado_logit["y_proba"]
acc = resultado_logit["acc"]
auc = resultado_logit["auc"]
cm = resultado_logit["cm"]

## 12. Coeficientes do Modelo Logístico

In [ ]:
coef_df, coef_df_sorted = gerar_tabela_e_fig_coeficientes(
    logit=logit,
    features_usadas=features_usadas,
    prefixo_arquivo="logit",
)

## 13. Treinamento e Avaliação - Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
)

rf.fit(dados_logit["X_train_scaled"], y_train)

rf_pred = rf.predict(dados_logit["X_test_scaled"])
rf_proba = rf.predict_proba(dados_logit["X_test_scaled"])[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_cm = confusion_matrix(y_test, rf_pred)

print(f"Acurácia Random Forest: {rf_acc:.3f}")
print(f"AUC ROC Random Forest: {rf_auc:.3f}")
print("\nMatriz de confusão:")
print(rf_cm)

print("\nRelatório de classificação:")
print(classification_report(y_test, rf_pred, digits=3))

## 14. Curva ROC - Random Forest

In [ ]:
fig_roc_rf, ax = plt.subplots(figsize=(6, 5))
fpr, tpr, th = roc_curve(y_test, rf_proba)
ax.plot(fpr, tpr, label=f"Random Forest (AUC = {rf_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("Falso positivo")
ax.set_ylabel("Verdadeiro positivo")
ax.legend()
plt.tight_layout()
save_figure(fig_roc_rf, "fig_curva_roc_random_forest.png")
plt.show()

## 15. Matriz de Confusão - Random Forest

In [ ]:
fig_rf_cm, ax_cm = plt.subplots(figsize=(6,5))
sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Não evadido (0)", "Evadido (1)"],
    yticklabels=["Não evadido (0)", "Evadido (1)"],
    ax=ax_cm,
)
ax_cm.set_xlabel("Classe predita")
ax_cm.set_ylabel("Classe verdadeira")
plt.tight_layout()
save_figure(fig_rf_cm, "fig_matriz_confusao_random_forest.png")
plt.show()

## 16. Importância das Variáveis - Random Forest

In [ ]:
importances = pd.Series(
    rf.feature_importances_,
    index=features_usadas
).sort_values(ascending=False)

fig_imp, ax = plt.subplots(figsize=(8,5))
importances.plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Importância (Gini importance)")
plt.tight_layout()
save_figure(fig_imp, "fig_importancia_variaveis_random_forest.png")
plt.show()

display(importances)